# 08c — Enron Spam/Ham Emails (English)

**C1 Source Type:** `Fichier de données (CSV / data file)`

---

## Objective

Load the `SetFit/enron_spam` dataset from HuggingFace Hub.
This email classification dataset provides English ham/spam email bodies
that serve as baseline comparison and input for cultural adaptation (EN → FR).

### Dataset Info

| Field | Value |
|-------|-------|
| Source | [HuggingFace: SetFit/enron_spam](https://huggingface.co/datasets/SetFit/enron_spam) |
| Format | Parquet via HuggingFace `datasets` library |
| Rows | ~33K |
| Labels | ham (0) / spam (1) |
| Language | English |

### Pipeline

```
HuggingFace Hub → Load with datasets → Normalize schema → Export CSV
```

### Output

- `data/raw/csv/en/enron_hamspam_<N>_<date>.csv`

In [1]:
# ── Imports & Constants ──────────────────────────────────────────────
from __future__ import annotations

import hashlib
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from datasets import load_dataset

# ── Configuration ────────────────────────────────────────────────────
OUTPUT_DIR: Path = Path("data/raw/csv/en")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_TEXT_LENGTH: int = 20
SCHEMA_COLS: list[str] = ["text", "label", "source", "language"]

print(f"Output dir   : {OUTPUT_DIR.resolve()}")
print(f"Min length   : {MIN_TEXT_LENGTH}")

Output dir   : /Users/michaeladebayo/Documents/Simplon/brief_projects/sicurre/data/raw/csv/en
Min length   : 20


## 1. Load Dataset

In [2]:
# ── Load SetFit/enron_spam from HuggingFace ──────────────────────────
print("Loading SetFit/enron_spam from HuggingFace...")

ds = load_dataset("SetFit/enron_spam", split="train")
source_name: str = "enron_spam"

df_raw: pd.DataFrame = ds.to_pandas()  # pyright: ignore[reportAssignmentType]

print(f"Source   : {source_name}")
print(f"Shape    : {df_raw.shape}")
print(f"Columns  : {list(df_raw.columns)}")
print(f"\nLabel distribution:")
print(df_raw["label"].value_counts())
df_raw.head(3)

Loading SetFit/enron_spam from HuggingFace...


README.md:   0%|          | 0.00/176 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl:   0%|          | 0.00/101M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/6.27M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/31716 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Source   : enron_spam
Shape    : (31716, 7)
Columns  : ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date']

Label distribution:
label
1    16163
0    15553
Name: count, dtype: int64


,message_id,text,label,label_text,subject,message,date
0,33214,any software just for 15 $ - 99 $ understandin...,1,spam,any software just for 15 $ - 99 $,understanding oem software\nlead me not into t...,2005-06-18
1,11929,perspective on ferc regulatory action client c...,0,ham,perspective on ferc regulatory action client c...,"19 th , 2 : 00 pm edt\nperspective on ferc reg...",2001-06-19
2,19784,wanted to try ci 4 lis but thought it was way ...,1,spam,wanted to try ci 4 lis but thought it was way ...,viagra at $ 1 . 12 per dose\nready to boost yo...,2004-09-11


## 2. Normalize Schema

In [3]:
# ── Normalize to unified schema ──────────────────────────────────────
text_col: str = "text" if "text" in df_raw.columns else df_raw.columns[0]

df_norm = pd.DataFrame({
    "text": df_raw[text_col].astype(str),
    "label": df_raw["label"].apply(
        lambda x: "spam" if str(x).lower() in ("spam", "1") else "ham"
    ),
    "source": source_name,
    "language": "en",
})

# Filter short texts
before: int = len(df_norm)
df_norm = df_norm[df_norm["text"].str.len() >= MIN_TEXT_LENGTH].reset_index(drop=True)
print(f"Before filter : {before:,}")
print(f"After filter  : {len(df_norm):,} (removed {before - len(df_norm)} short texts)")
print(f"\nLabel distribution:")
print(df_norm["label"].value_counts())

Before filter : 31,716
After filter  : 31,621 (removed 95 short texts)

Label distribution:
label
spam    16079
ham     15542
Name: count, dtype: int64


## 3. Deduplicate

In [4]:
# ── Deduplication by text hash ────────────────────────────────────────
before = len(df_norm)
df_norm["text_hash"] = df_norm["text"].str[:300].apply(
    lambda t: hashlib.sha256(t.encode("utf-8", errors="ignore")).hexdigest()
)
df_norm = (
    df_norm.drop_duplicates(subset="text_hash", keep="first")
    .drop(columns="text_hash")
    .reset_index(drop=True)
)
after: int = len(df_norm)
print(f"Before dedup : {before:,}")
print(f"After dedup  : {after:,}")
print(f"Removed      : {before - after:,}")

Before dedup : 31,621
After dedup  : 28,191
Removed      : 3,430


## 4. Export

In [5]:
# ── Export ─────────────────────────────────────────────────────────────
timestamp: str = datetime.now(timezone.utc).strftime("%Y%m%d")
n_rows: int = len(df_norm)

if n_rows > 0:
    filename: str = f"enron_hamspam_{n_rows}_{timestamp}.csv"
    output_path: Path = OUTPUT_DIR / filename
    df_norm.to_csv(output_path, index=False, encoding="utf-8")

    size_mb: float = output_path.stat().st_size / (1024 * 1024)
    print(f"Exported  : {output_path}")
    print(f"Rows      : {n_rows:,}")
    print(f"Size      : {size_mb:.2f} MB")
    print(f"Columns   : {list(df_norm.columns)}")
else:
    print("Nothing to export.")

Exported  : data/raw/csv/en/enron_hamspam_28191_20260301.csv
Rows      : 28,191
Size      : 40.74 MB
Columns   : ['text', 'label', 'source', 'language']


## 5. Summary

| Criterion | Evidence |
|-----------|----------|
| **Source type** | Fichier de données — Parquet via HuggingFace Hub |
| **Provider** | `SetFit/enron_spam` |
| **Content** | English ham/spam email bodies |
| **Schema** | Normalized to `(text, label, source, language)` |
| **Deduplication** | SHA-256 hash of first 300 chars |

### Role in Pipeline

English sources serve as:
1. **Input for cultural adaptation** (EN → FR via LLM in notebook 10)
2. **Baseline comparison** for model evaluation
3. **Pattern extraction** for synthetic French phishing generation